In [1]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd 
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


In [2]:
root = Path("./data/mallorn-astronomical-classification-challenge/")

# 1) Global training metadata with labels
train_log = pd.read_csv(root / "train_log.csv")
test_log = pd.read_csv(root / "test_log.csv")

# 2) Load one split's lightcurves and attach labels/features from global log
def load_split(split_name: str, is_test=False):
    if is_test:
        lc_path = root / split_name / "test_full_lightcurves.csv"
    else:
        lc_path = root / split_name / "train_full_lightcurves.csv"
    lc = pd.read_csv(lc_path)

    log = test_log if is_test else train_log

    # Keep only metadata rows for this split
    meta = log.loc[log["split"] == split_name].copy()

    # Merge target + static features onto each observation row
    merged = lc.merge(
        meta[["object_id", "target", "Z", "Z_err", "EBV", "SpecType", "split"]],
        on="object_id",
        how="left",
        validate="many_to_one"
    )
    return merged, meta

In [ ]:
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTER_ORDER)}

def encode_sequence(obj_df):
    obj_df = obj_df.sort_values("Time (MJD)").copy()
    t = obj_df["Time (MJD)"].to_numpy(dtype=np.float32)
    flux = obj_df["Flux"].to_numpy(dtype=np.float32)
    flux_err = obj_df["Flux_err"].to_numpy(dtype=np.float32)

    dt = np.diff(t, prepend=t[0]).astype(np.float32)

    filt_oh = np.zeros((len(obj_df), len(FILTER_ORDER)), dtype=np.float32)
    filt_idx = obj_df["Filter"].map(FILTER_TO_IDX).to_numpy()
    valid = ~pd.isna(filt_idx)
    filt_oh[np.arange(len(obj_df))[valid], filt_idx[valid].astype(int)] = 1.0

    x = np.column_stack([flux, flux_err, dt, filt_oh]).astype(np.float32)
    return torch.tensor(x, dtype=torch.float32)

class LightCurveSplitDataset(Dataset):
    def __init__(self, split_name, root_path, is_test=False):
        self.split_name = split_name
        self.root_path = root_path
        self.is_test = is_test

        self.merged, self.meta = load_split(split_name, is_test=is_test)

        self.meta = self.meta.set_index("object_id", drop=False)

        object_set = set(self.merged["object_id"].unique())
        self.object_ids = [oid for oid in self.meta["object_id"].tolist() if oid in object_set]
        self.grouped = dict(tuple(self.merged.groupby("object_id")))

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        oid = self.object_ids[idx]
        obj_df = self.grouped[oid]
        m = self.meta.loc[oid]

        seq = encode_sequence(obj_df) 

        z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
        z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
        ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
        static = torch.tensor([z, z_err, ebv], dtype=torch.float32)

        y = torch.tensor(float(m["target"]), dtype=torch.float32)

        return {
            "object_id": oid,
            "seq": seq,
            "static": static,
            "y": y
        }

"""
Build one mini-batch from a list of dataset samples.

This function:
- collects variable-length sequence tensors (`seq`)
- records each sequence length (`lengths`)
- pads sequences to the max length in the batch (`seq_padded`)
- stacks fixed-size static features (`static`) and labels (`y`)
- keeps object IDs as a Python list (`object_id`)

Returned shapes:
- seq: [B, T_max, F]
- lengths: [B]
- static: [B, 3]
- y: [B]
"""
def collate_pad(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
        "y": y
    }

# Example usage:
train_ds = LightCurveSplitDataset("split_01", root)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)

batch = next(iter(train_loader))
# print("seq shape:", batch["seq"].shape)        # [B, T_max, F]
# print("lengths:", batch["lengths"][:8])
# print("static shape:", batch["static"].shape)  # [B, 3]
# print("y shape:", batch["y"].shape)            # [B]

In [4]:
class LSTMWithStatic(nn.Module):
    def __init__(self, seq_input_size=9, static_input_size=3, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.static_net = nn.Sequential(
            nn.Linear(static_input_size, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size + 16, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, seq, lengths, static):
        packed = nn.utils.rnn.pack_padded_sequence(
            seq, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        seq_repr = h_n[-1]
        static_repr = self.static_net(static)
        x = torch.cat([seq_repr, static_repr], dim=1)
        logits = self.head(x).squeeze(1)
        return logits

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMWithStatic(seq_input_size=9, static_input_size=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

split_names = sorted(train_log["split"].dropna().unique().tolist())
print("Training splits:", split_names)

n_epochs = 10
for epoch in range(1, n_epochs + 1):
    model.train()
    running_loss = 0.0
    n_obs = 0

    for split_name in split_names:
        ds = LightCurveSplitDataset(split_name, root)
        loader = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_pad)

        for batch in loader:
            seq = batch["seq"].to(device)
            lengths = batch["lengths"]
            static = batch["static"].to(device)
            y = batch["y"].to(device)

            logits = model(seq, lengths, static)
            loss = loss_fn(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            bs = y.size(0)
            running_loss += loss.item() * bs
            n_obs += bs

    epoch_loss = running_loss / max(n_obs, 1)

    if epoch == 1 or epoch % 2 == 0:
        model.eval()
        with torch.no_grad():
            logits_all = []
            y_all = []
            for split_name in split_names:
                ds = LightCurveSplitDataset(split_name, root)
                loader = DataLoader(ds, batch_size=128, shuffle=False, collate_fn=collate_pad)
                for batch in loader:
                    seq = batch["seq"].to(device)
                    lengths = batch["lengths"]
                    static = batch["static"].to(device)
                    y = batch["y"].to(device)

                    logits = model(seq, lengths, static)
                    logits_all.append(logits.cpu())
                    y_all.append(y.cpu())

            logits_all = torch.cat(logits_all)
            y_all = torch.cat(y_all)
            probs = torch.sigmoid(logits_all)
            preds = (probs >= 0.5).float()
            acc = (preds == y_all).float().mean().item()

        print(f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | train_acc={acc:.4f}")

Training splits: ['split_01', 'split_02', 'split_03', 'split_04', 'split_05', 'split_06', 'split_07', 'split_08', 'split_09', 'split_10', 'split_11', 'split_12', 'split_13', 'split_14', 'split_15', 'split_16', 'split_17', 'split_18', 'split_19', 'split_20']
Epoch 01 | loss=nan | train_acc=0.9514
Epoch 02 | loss=nan | train_acc=0.9514
Epoch 04 | loss=nan | train_acc=0.9514
Epoch 06 | loss=nan | train_acc=0.9514
Epoch 08 | loss=nan | train_acc=0.9514
Epoch 10 | loss=nan | train_acc=0.9514
